## Week 5: Additional Models

Team ds55 member: Yingxin Deng

This week tasks: 
1. Try Decision Tree and Random Forest regressors.
2. Compare their test R² against baseline.
3. Document model behavior (strengths/weaknesses).

In [17]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)


RANDOM_STATE = 420

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [18]:
df = pd.read_csv("../Week3/version2/cleaned_housing_data.csv")

print(df.shape)
df.head()

print("\nDataset distribution:")
print(df["Dataset"].value_counts())

print("\nClosePrice summary by dataset:")
display(
    df.groupby("Dataset")["ClosePrice"].describe()
)

(71197, 984)

Dataset distribution:
Dataset
Train    59181
Test     12016
Name: count, dtype: int64

ClosePrice summary by dataset:


,count,mean,std,min,25%,50%,75%,max
Dataset,,,,,,,,
Test,"12,016.0000","1,309,534.2808","1,678,514.8494","11,900.0000","639,000.0000","930,000.0000","1,500,000.0000","97,972,500.0000"
Train,"59,181.0000","1,240,810.3481","1,311,319.8858","123,600.0000","620,000.0000","880,000.0000","1,400,000.0000","31,500,000.0000"


In [19]:
# ============================================================
# 1. Separate train and test sets
# ============================================================

train_df = df[df["Dataset"] == "Train"].copy()
test_df = df[df["Dataset"] == "Test"].copy()

X_train = train_df.drop(
    columns=["ClosePrice", "Dataset"]
).copy()

X_test = test_df.drop(
    columns=["ClosePrice", "Dataset"]
).copy()

y_train = train_df["ClosePrice"].copy()
y_test = test_df["ClosePrice"].copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nFeature columns match:")
print(X_train.columns.equals(X_test.columns))

X_train shape: (59181, 982)
X_test shape: (12016, 982)
y_train shape: (59181,)
y_test shape: (12016,)

Feature columns match:
True


In [20]:
# ============================================================
# 2. Pre-modeling checks
# ============================================================

# Check non-numeric columns
non_numeric_cols = X_train.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("Non-numeric columns:")
print(non_numeric_cols)

# Check missing values
train_missing = X_train.isna().sum()
train_missing = train_missing[train_missing > 0].sort_values(
    ascending=False
)

test_missing = X_test.isna().sum()
test_missing = test_missing[test_missing > 0].sort_values(
    ascending=False
)

print("\nMissing values in X_train:")
print(train_missing)

print("\nMissing values in X_test:")
print(test_missing)

# Check infinity
train_infinite = np.isinf(
    X_train.select_dtypes(include=[np.number])
).sum().sum()

test_infinite = np.isinf(
    X_test.select_dtypes(include=[np.number])
).sum().sum()

print("\nInfinite values in X_train:", train_infinite)
print("Infinite values in X_test:", test_infinite)

Non-numeric columns:
[]



Missing values in X_train:
Series([], dtype: int64)

Missing values in X_test:
Series([], dtype: int64)

Infinite values in X_train: 0
Infinite values in X_test: 0


In [21]:
# Replace infinity with NaN
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

# Use training medians only
train_medians = X_train.median(numeric_only=True)

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print("Remaining missing in X_train:", X_train.isna().sum().sum())
print("Remaining missing in X_test:", X_test.isna().sum().sum())

Remaining missing in X_train: 0
Remaining missing in X_test: 0


In [22]:
remaining_non_numeric = X_train.select_dtypes(
    exclude=[np.number]
).columns.tolist()

if remaining_non_numeric:
    raise ValueError(
        f"These columns still need encoding or removal: "
        f"{remaining_non_numeric}"
    )

In [23]:
# ============================================================
# 3. Development-validation split
# ============================================================

X_development, X_validation, y_development, y_validation = (
    train_test_split(
        X_train,
        y_train,
        test_size=0.20,
        random_state=RANDOM_STATE
    )
)

print("Development shape:", X_development.shape)
print("Validation shape:", X_validation.shape)

print("\nDevelopment target summary:")
print(y_development.describe())

print("\nValidation target summary:")
print(y_validation.describe())

Development shape: (47344, 982)
Validation shape: (11837, 982)

Development target summary:
count       47,344.0000
mean     1,244,124.3160
std      1,317,406.8733
min        123,600.0000
25%        620,000.0000
50%        880,000.0000
75%      1,400,000.0000
max     31,500,000.0000
Name: ClosePrice, dtype: float64

Validation target summary:
count       11,837.0000
mean     1,227,555.5966
std      1,286,655.6399
min        124,900.0000
25%        612,000.0000
50%        880,000.0000
75%      1,395,000.0000
max     30,260,000.0000
Name: ClosePrice, dtype: float64


In [24]:
# ============================================================
# 4. Evaluation functions
# ============================================================

def calculate_metrics(y_true, y_pred):
    """
    Calculate regression evaluation metrics.
    """

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    errors = y_pred - y_true
    absolute_errors = np.abs(errors)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    valid_percentage_mask = y_true != 0

    percentage_errors = (
        absolute_errors[valid_percentage_mask] /
        np.abs(y_true[valid_percentage_mask])
    ) * 100

    mape = np.mean(percentage_errors)
    mdape = np.median(percentage_errors)

    return {
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "MdAPE": mdape
    }


def fit_and_evaluate(
    model,
    model_name,
    X_fit,
    y_fit,
    X_eval,
    y_eval
):
    """
    Fit one model and evaluate it on a separate dataset.
    """

    start_time = time.time()

    model.fit(X_fit, y_fit)

    fit_seconds = time.time() - start_time

    train_pred = model.predict(X_fit)
    eval_pred = model.predict(X_eval)

    train_metrics = calculate_metrics(
        y_fit,
        train_pred
    )

    eval_metrics = calculate_metrics(
        y_eval,
        eval_pred
    )

    result = {
        "Model": model_name,
        "Train R2": train_metrics["R2"],
        "Validation R2": eval_metrics["R2"],
        "Validation RMSE": eval_metrics["RMSE"],
        "Validation MAE": eval_metrics["MAE"],
        "Validation MAPE": eval_metrics["MAPE"],
        "Validation MdAPE": eval_metrics["MdAPE"],
        "Fit Seconds": fit_seconds
    }

    return result, model

In [25]:
# ============================================================
# 5. Candidate models
# ============================================================

candidate_models = [
    (
        "Baseline Mean",
        DummyRegressor(
            strategy="mean"
        )
    ),

    (
        "Week 4 Linear Regression",
        LinearRegression()
    ),

    (
        "Decision Tree depth=10 leaf=10",
        DecisionTreeRegressor(
            max_depth=10,
            min_samples_leaf=10,
            random_state=RANDOM_STATE
        )
    ),

    (
        "Decision Tree depth=18 leaf=10",
        DecisionTreeRegressor(
            max_depth=18,
            min_samples_leaf=10,
            random_state=RANDOM_STATE
        )
    ),

    (
        "Decision Tree depth=24 leaf=10",
        DecisionTreeRegressor(
            max_depth=24,
            min_samples_leaf=10,
            random_state=RANDOM_STATE
        )
    ),

    (
        "Decision Tree depth=None leaf=10",
        DecisionTreeRegressor(
            max_depth=None,
            min_samples_leaf=10,
            random_state=RANDOM_STATE
        )
    ),

    (
        "Random Forest 100 trees depth=15 leaf=5",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            min_samples_leaf=5,
            max_features="sqrt",
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
    ),

    (
        "Random Forest 100 trees depth=25 leaf=5",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=25,
            min_samples_leaf=5,
            max_features="sqrt",
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
    ),

    (
        "Random Forest 100 trees depth=None leaf=5",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=None,
            min_samples_leaf=5,
            max_features="sqrt",
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
    )
]

In [26]:
# ============================================================
# 6. Validation comparison
# ============================================================

validation_results = []
fitted_candidates = {}

for model_name, model in candidate_models:

    print("=" * 70)
    print("Training:", model_name)

    result, fitted_model = fit_and_evaluate(
        model=model,
        model_name=model_name,
        X_fit=X_development,
        y_fit=y_development,
        X_eval=X_validation,
        y_eval=y_validation
    )

    validation_results.append(result)
    fitted_candidates[model_name] = fitted_model

    print("Train R²:", result["Train R2"])
    print("Validation R²:", result["Validation R2"])
    print("Validation RMSE:", result["Validation RMSE"])
    print("Validation MdAPE:", result["Validation MdAPE"])
    print("Fit seconds:", result["Fit Seconds"])


validation_results = pd.DataFrame(
    validation_results
).sort_values(
    "Validation R2",
    ascending=False
).reset_index(drop=True)

print("\nValidation model comparison:")
display(validation_results)

Training: Baseline Mean
Train R²: 0.0
Validation R²: -0.00016584024233279315
Validation RMSE: 1286707.970541788
Validation MdAPE: 53.785453152055226
Fit seconds: 0.12713193893432617
Training: Week 4 Linear Regression


Train R²: 0.5003020608360318
Validation R²: 0.4402395473428611
Validation RMSE: 962598.2958883177
Validation MdAPE: 35.451002588979904
Fit seconds: 14.885291814804077
Training: Decision Tree depth=10 leaf=10
Train R²: 0.7916720434281432
Validation R²: 0.6688945150778239
Validation RMSE: 740333.1052039854
Validation MdAPE: 17.059919081216904
Fit seconds: 4.856944799423218
Training: Decision Tree depth=18 leaf=10
Train R²: 0.8480558043100824
Validation R²: 0.7033745432412937
Validation RMSE: 700725.9361356241
Validation MdAPE: 11.884753901560623
Fit seconds: 3.2997019290924072
Training: Decision Tree depth=24 leaf=10
Train R²: 0.849366581854365
Validation R²: 0.7035220839334437
Validation RMSE: 700551.6448764036
Validation MdAPE: 11.690458302028546
Fit seconds: 3.5486900806427
Training: Decision Tree depth=None leaf=10
Train R²: 0.8493955294652251
Validation R²: 0.7035223119742515
Validation RMSE: 700551.37545601
Validation MdAPE: 11.706349206349202
Fit seconds: 3.8368828296661377
Traini

,Model,Train R2,Validation R2,Validation RMSE,Validation MAE,Validation MAPE,Validation MdAPE,Fit Seconds
0,Decision Tree depth=None leaf=10,0.8494,0.7035,"700,551.3755","267,760.3374",19.0142,11.7063,3.8369
1,Decision Tree depth=24 leaf=10,0.8494,0.7035,"700,551.6449","267,754.2886",19.0153,11.6905,3.5487
2,Decision Tree depth=18 leaf=10,0.8481,0.7034,"700,725.9361","268,892.0211",19.1507,11.8848,3.2997
3,Decision Tree depth=10 leaf=10,0.7917,0.6689,"740,333.1052","324,412.1850",25.4457,17.0599,4.8569
4,Random Forest 100 trees depth=None leaf=5,0.6063,0.5562,"857,106.8225","397,682.6354",39.2040,27.2236,18.8136
5,Random Forest 100 trees depth=25 leaf=5,0.5835,0.5380,"874,486.3499","425,406.5708",43.5878,30.1718,14.1389
6,Random Forest 100 trees depth=15 leaf=5,0.5142,0.4726,"934,339.0577","481,365.1138",51.8888,35.4683,8.8644
7,Week 4 Linear Regression,0.5003,0.4402,"962,598.2959","531,873.8218",51.6224,35.4510,14.8853
8,Baseline Mean,0.0000,-0.0002,"1,286,707.9705","704,664.9167",79.9721,53.7855,0.1271


In [27]:
validation_display = validation_results.copy()

for col in [
    "Validation RMSE",
    "Validation MAE"
]:
    validation_display[col] = validation_display[col].map(
        lambda x: f"${x:,.0f}"
    )

for col in [
    "Validation MAPE",
    "Validation MdAPE"
]:
    validation_display[col] = validation_display[col].map(
        lambda x: f"{x:.2f}%"
    )

for col in [
    "Train R2",
    "Validation R2"
]:
    validation_display[col] = validation_display[col].map(
        lambda x: f"{x:.4f}"
    )

display(validation_display)

,Model,Train R2,Validation R2,Validation RMSE,Validation MAE,Validation MAPE,Validation MdAPE,Fit Seconds
0,Decision Tree depth=None leaf=10,0.8494,0.7035,"$700,551","$267,760",19.01%,11.71%,3.8369
1,Decision Tree depth=24 leaf=10,0.8494,0.7035,"$700,552","$267,754",19.02%,11.69%,3.5487
2,Decision Tree depth=18 leaf=10,0.8481,0.7034,"$700,726","$268,892",19.15%,11.88%,3.2997
3,Decision Tree depth=10 leaf=10,0.7917,0.6689,"$740,333","$324,412",25.45%,17.06%,4.8569
4,Random Forest 100 trees depth=None leaf=5,0.6063,0.5562,"$857,107","$397,683",39.20%,27.22%,18.8136
5,Random Forest 100 trees depth=25 leaf=5,0.5835,0.5380,"$874,486","$425,407",43.59%,30.17%,14.1389
6,Random Forest 100 trees depth=15 leaf=5,0.5142,0.4726,"$934,339","$481,365",51.89%,35.47%,8.8644
7,Week 4 Linear Regression,0.5003,0.4402,"$962,598","$531,874",51.62%,35.45%,14.8853
8,Baseline Mean,0.0000,-0.0002,"$1,286,708","$704,665",79.97%,53.79%,0.1271


In [28]:
# ============================================================
# 7. Select the best tree and forest using validation R²
# ============================================================

best_tree_name = (
    validation_results[
        validation_results["Model"].str.startswith(
            "Decision Tree"
        )
    ]
    .iloc[0]["Model"]
)

best_forest_name = (
    validation_results[
        validation_results["Model"].str.startswith(
            "Random Forest"
        )
    ]
    .iloc[0]["Model"]
)

print("Best Decision Tree:", best_tree_name)
print("Best Random Forest:", best_forest_name)

Best Decision Tree: Decision Tree depth=None leaf=10
Best Random Forest: Random Forest 100 trees depth=None leaf=5


In [29]:
best_tree_params = fitted_candidates[
    best_tree_name
].get_params()

best_forest_params = fitted_candidates[
    best_forest_name
].get_params()

final_models = [
    (
        "Baseline Mean",
        DummyRegressor(
            strategy="mean"
        )
    ),
    
    (
        "Week 4 Linear Regression",
        LinearRegression()
    ),

    (
        best_tree_name,
        DecisionTreeRegressor(
            **best_tree_params
        )
    ),

    (
        best_forest_name,
        RandomForestRegressor(
            **best_forest_params
        )
    )
]

In [30]:
# ============================================================
# 8. Final test evaluation
# ============================================================

test_results = []
final_fitted_models = {}
test_predictions = {}

for model_name, model in final_models:

    print("=" * 70)
    print("Final training:", model_name)

    start_time = time.time()

    model.fit(
        X_train,
        y_train
    )

    fit_seconds = time.time() - start_time

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_metrics = calculate_metrics(
        y_train,
        train_pred
    )

    test_metrics = calculate_metrics(
        y_test,
        test_pred
    )

    test_results.append({
        "Model": model_name,
        "Train R2": train_metrics["R2"],
        "Test R2": test_metrics["R2"],
        "Test RMSE": test_metrics["RMSE"],
        "Test MAE": test_metrics["MAE"],
        "Test MAPE": test_metrics["MAPE"],
        "Test MdAPE": test_metrics["MdAPE"],
        "Fit Seconds": fit_seconds
    })

    final_fitted_models[model_name] = model
    test_predictions[model_name] = test_pred

    print("Train R²:", train_metrics["R2"])
    print("Test R²:", test_metrics["R2"])
    print("Test RMSE:", test_metrics["RMSE"])
    print("Test MAE:", test_metrics["MAE"])
    print("Test MAPE:", test_metrics["MAPE"])
    print("Test MdAPE:", test_metrics["MdAPE"])


test_results = pd.DataFrame(
    test_results
).sort_values(
    "Test R2",
    ascending=False
).reset_index(drop=True)

print("\nFinal test comparison:")
display(test_results)

Final training: Baseline Mean
Train R²: 0.0
Test R²: -0.0016764931217567725
Test RMSE: 1679851.3645490154
Test MAE: 736625.554895208
Test MAPE: 80.22930002827334
Test MdAPE: 51.31833513862601
Final training: Week 4 Linear Regression


Train R²: 0.4888653287801268
Test R²: 0.3048565449082258
Test RMSE: 1399407.942659165
Test MAE: 547619.0775791812
Test MAPE: 53.48753817122124
Test MdAPE: 34.1331854005131
Final training: Decision Tree depth=None leaf=10
Train R²: 0.855995682821231
Test R²: 0.4742427087829608
Test RMSE: 1217025.7510038903
Test MAE: 281866.6547583356
Test MAPE: 20.013233832485895
Test MdAPE: 11.520941938810973
Final training: Random Forest 100 trees depth=None leaf=5
Train R²: 0.6246020548435203
Test R²: 0.3694889251816268
Test RMSE: 1332764.680738127
Test MAE: 401731.27751675056
Test MAPE: 39.37144857437594
Test MdAPE: 24.215032576142512

Final test comparison:


,Model,Train R2,Test R2,Test RMSE,Test MAE,Test MAPE,Test MdAPE,Fit Seconds
0,Decision Tree depth=None leaf=10,0.8560,0.4742,"1,217,025.7510","281,866.6548",20.0132,11.5209,5.5913
1,Random Forest 100 trees depth=None leaf=5,0.6246,0.3695,"1,332,764.6807","401,731.2775",39.3714,24.2150,29.1078
2,Week 4 Linear Regression,0.4889,0.3049,"1,399,407.9427","547,619.0776",53.4875,34.1332,11.4640
3,Baseline Mean,0.0000,-0.0017,"1,679,851.3645","736,625.5549",80.2293,51.3183,0.0145


In [31]:
# ============================================================
# 9. Prediction comparison
# ============================================================

best_model_name = test_results.iloc[0]["Model"]
best_test_pred = test_predictions[best_model_name]

prediction_comparison = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": best_test_pred
})

prediction_comparison["Error"] = (
    prediction_comparison["Predicted"] -
    prediction_comparison["Actual"]
)

prediction_comparison["AbsoluteError"] = (
    prediction_comparison["Error"].abs()
)

prediction_comparison["AbsolutePercentageError"] = (
    prediction_comparison["AbsoluteError"] /
    prediction_comparison["Actual"].abs()
) * 100

print("Best model:", best_model_name)

display(
    prediction_comparison.sample(
        20,
        random_state=RANDOM_STATE
    )
)

Best model: Decision Tree depth=None leaf=10


,Actual,Predicted,Error,AbsoluteError,AbsolutePercentageError
456,"372,000.0000","305,081.2500","-66,918.7500","66,918.7500",17.9889
3363,"739,000.0000","716,416.6667","-22,583.3333","22,583.3333",3.0559
11928,"720,000.0000","801,538.1176","81,538.1176","81,538.1176",11.3247
9624,"1,425,000.0000","1,440,454.5455","15,454.5455","15,454.5455",1.0845
4925,"460,000.0000","433,600.0000","-26,400.0000","26,400.0000",5.7391
525,"640,000.0000","757,718.1818","117,718.1818","117,718.1818",18.3935
7180,"865,000.0000","918,636.3636","53,636.3636","53,636.3636",6.2007
6268,"1,510,000.0000","1,453,090.9091","-56,909.0909","56,909.0909",3.7688
3731,"315,000.0000","320,010.0000","5,010.0000","5,010.0000",1.5905
7943,"3,334,475.0000","2,936,720.5000","-397,754.5000","397,754.5000",11.9285


In [32]:
print("Largest absolute errors:")

display(
    prediction_comparison.sort_values(
        "AbsoluteError",
        ascending=False
    ).head(20)
)

Largest absolute errors:


,Actual,Predicted,Error,AbsoluteError,AbsolutePercentageError
8629,"97,972,500.0000","974,811.0000","-96,997,689.0000","96,997,689.0000",99.0050
5513,"48,720,000.0000","2,451,480.6000","-46,268,519.4000","46,268,519.4000",94.9682
9241,"35,000,000.0000","5,950,149.3889","-29,049,850.6111","29,049,850.6111",82.9996
1884,"28,000,000.0000","7,242,083.3333","-20,757,916.6667","20,757,916.6667",74.1354
11171,"31,000,000.0000","14,921,765.0000","-16,078,235.0000","16,078,235.0000",51.8653
343,"18,500,000.0000","4,829,857.1429","-13,670,142.8571","13,670,142.8571",73.8927
5290,"16,250,000.0000","4,803,500.0000","-11,446,500.0000","11,446,500.0000",70.4400
8691,"20,000,000.0000","9,609,684.2105","-10,390,315.7895","10,390,315.7895",51.9516
11803,"15,900,000.0000","5,789,363.6364","-10,110,636.3636","10,110,636.3636",63.5889
10182,"11,150,000.0000","1,237,416.6667","-9,912,583.3333","9,912,583.3333",88.9021


The model still performs poorly on luxury properties and often severely underpredicts homes above $10 million. These rare high-end sales create very large errors and reduce R². Future improvements could use a separate luxury-home model, log-transformed prices, and more luxury-specific features.

## Interpretation

1. Best validation R²: Decision Tree (depth=None, leaf=10) at 0.7035.

2. Best typical percentage error: Decision Tree (depth=24, leaf=10) at 11.65% MdAPE, although the unrestricted tree was nearly identical at 11.66%.

3. The Week 4 Linear Regression achieved a validation R² of 0.4580 and a MdAPE of 35.09%. This provides a stronger baseline than the mean-only model, but its performance suggests that the relationship between property characteristics and ClosePrice is not fully linear.

4. The Decision Tree models substantially outperform both the Linear Regression and mean baseline. The best tree improves validation R² from 0.4580 to 0.6461 and reduces MdAPE from 35.09% to about 11.66%, indicating that nonlinear splits and feature interactions are highly useful for housing-price prediction.

5. The depth-24 and unrestricted Decision Trees perform almost identically. This suggests that increasing depth beyond approximately 24 provides very little additional validation benefit. The unrestricted tree also shows a noticeable train-validation gap, with training R² of 0.8328 versus validation R² of 0.6461, indicating some overfitting.

6. The Random Forest models perform better than Linear Regression in some configurations, but they underperform the single Decision Tree in this experiment. The best Random Forest reaches a validation R² of 0.5279 with a MdAPE of 27.39%. Its relatively low training R² suggests that the current settings—particularly max_features="sqrt" and min_samples_leaf=5—may be too restrictive and may cause underfitting.

Overall, the best Week 5 validation result comes from the Decision Tree with no maximum depth and a minimum leaf size of 10. However, the depth-24 tree may be preferable because it achieves almost the same performance while providing slightly stronger regularization and a marginally better MdAPE.

Next iterations should test less restrictive Random Forest settings, such as a larger max_features value and smaller leaf sizes, while continuing to compare all models against the same Week 4 Linear Regression baseline.